In [ ]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Load data
data = pd.read_csv("ret_sample.csv")

# Keep year column before dropping
year_col = data["year"].copy()

# Drop irrelevant columns
drop_columns = ['id', 'date', 'ret_eom', 'gvkey', 'lid', 'excntry', 
                'year', 'month', 'chr_date', 'char_eom']
data = data.drop(columns=[c for c in drop_columns if c in data.columns])

# Standardize features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(data)

# Reattach year info
data_scaled = pd.DataFrame(scaled_features, columns=data.columns)
data_scaled["year"] = year_col.values

# HMM parameters
n_hidden_states = 3

# Dictionary to store results
results = {}

for yr in sorted(data_scaled["year"].unique()):
    yearly_data = data_scaled[data_scaled["year"] == yr].drop(columns="year").values
    
    if len(yearly_data) < n_hidden_states:  # Skip years with too few samples
        continue
    
    # Train HMM on this year
    model = GaussianHMM(n_components=n_hidden_states, 
                        covariance_type="diag", 
                        n_iter=1000,
                        random_state=42)
    
    model.fit(yearly_data)
    hidden_states = model.predict(yearly_data)
    
    # Evaluate log-likelihood
    log_likelihood = model.score(yearly_data)
    
    # Store results
    results[yr] = {
        "model": model,
        "hidden_states": hidden_states,
        "log_likelihood": log_likelihood
    }
    
    print(f"Year {yr}: log-likelihood = {log_likelihood:.2f}")

    # Visualization for each year
    plt.figure(figsize=(12,4))
    plt.plot(yearly_data[:,0], label="Feature 1")
    plt.scatter(range(len(hidden_states)), hidden_states, 
                c=hidden_states, cmap="viridis", s=15)
    plt.title(f"Hidden States for {yr}")
    plt.legend()
    plt.show()

# Example: Access states for 2015
if 2015 in results:
    print("2015 states:", results[2015]["hidden_states"])
